# CAMELS-US 2026 — Final Top-18 vs Top-10 / Hydrology-Informed Analysis

**Run All notebook** for the final Top-18 control experiment.

This notebook audits the completed 80 Add-8 tests, builds compact raw Top-18 predictions, creates Top18-median and Top18-mean ensembles, validates the metric protocol against the existing Top-10 results, and produces manuscript-ready comparisons for overall test performance, water years, seasons, hydrological clusters (when a defensible basin→cluster map is found), and high-flow events using the existing event catalog.

### Locked rules
- `D:\Datesets\CAMELS_US` was used only for NeuralHydrology inference; inference is already finished.
- **Canonical observations for all analysis come from `Pickles\Obs_test.p`.**
- Primary comparison: **Top18-median vs Top10-median**.
- Top18-mean is secondary.
- Existing Pickles are read-only; all new outputs go to `Revision_2026_Top18_4Framework\Top18_vs_Top10_Final_Analysis`.
- The notebook reproduces the existing Top-10 metric protocol before calculating Top-18. If it cannot reproduce it closely, it stops rather than silently changing definitions.
- Exact normalized-event-bin reproduction is not fabricated. It is only generated if the original binning logic is defensibly recoverable from the available artifacts.


In [2]:
from pathlib import Path

PROJECT = Path(r"F:\Experiments\CAMELS_US\Clean_4_upload")
REVISION = PROJECT / "Revision_2026_Top18_4Framework"
RUN_DIR = REVISION / "runs" / "top18_additional8_selected_epoch_train_v57"
PICKLES = PROJECT / "Pickles"

OBS_P = PICKLES / "Obs_test.p"
RAW_TOP10_P = PICKLES / "All_Top10_Configs_CAMELSUS_test.p"
ENS_TOP10_P = PICKLES / "ensemble_Top10_CAMELSUS_test.p"
ENS_CLUSTER_P = PICKLES / "ensemble_Cluster_wise_CAMELSUS_test.p"
OLD_METRICS_OVERALL_P = PICKLES / "metrics_all_test_period.p"
OLD_METRICS_WY_P = PICKLES / "metrics_by_water_year.p"
OLD_METRICS_SEASON_P = PICKLES / "metrics_by_year_season.p"
OLD_HIGHFLOW_EVENTWISE_P = PICKLES / "metrics_highflow_events_eventwise.p"
OLD_HIGHFLOW_BINS_P = PICKLES / "metrics_highflow_events_normalized_bins.p"
HIGHFLOW_LOG = PICKLES / "highflow_events_log_with_global.csv"
GLOBAL_EVENT_CATALOG = PICKLES / "global_event_catalog.csv"

OUT = REVISION / "Top18_vs_Top10_Final_Analysis"
OUT_PICKLES = OUT / "pickles"
OUT_TABLES = OUT / "tables"
OUT_FIGURES = OUT / "figures"
OUT_CACHE = OUT / "cache"
OUT_AUDIT = OUT / "audit"
for p in [OUT, OUT_PICKLES, OUT_TABLES, OUT_FIGURES, OUT_CACHE, OUT_AUDIT]:
    p.mkdir(parents=True, exist_ok=True)

PRIMARY_TOP18_LABEL = "Top18"
SECONDARY_TOP18_LABEL = "Top18-mean"
CALCULATE_TOP18_MEAN_OVERALL = True
WRITE_COMPACT_RAW_ADD8 = True
WRITE_COMPACT_RAW_TOP18 = True
KEY_METRICS = ["NSE","KGE","RMSE","FHV","FMS","FLV","Peak-Timing","Peak-MAPE","Missed-Peaks","MAPE"]

print("Output root:", OUT)


Output root: F:\Experiments\CAMELS_US\Clean_4_upload\Revision_2026_Top18_4Framework\Top18_vs_Top10_Final_Analysis


In [3]:
import gc, json, math, pickle, re, shutil, subprocess, sys, warnings
from collections import defaultdict
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
from scipy.stats import wilcoxon
NH_SOURCE = Path(r"C:\Users\hosseinifh\neuralhydrology")
if not NH_SOURCE.exists():
    raise FileNotFoundError(f"NeuralHydrology checkout not found: {NH_SOURCE}")

if str(NH_SOURCE) not in sys.path:
    sys.path.insert(0, str(NH_SOURCE))


from neuralhydrology.evaluation import metrics as nh_metrics


warnings.filterwarnings("ignore", category=FutureWarning)

def read_pickle_compat(path: Path):
    return pd.read_pickle(path)

required = [RUN_DIR, PICKLES, OBS_P, RAW_TOP10_P, ENS_TOP10_P,
            OLD_METRICS_OVERALL_P, OLD_METRICS_WY_P, OLD_METRICS_SEASON_P]
missing = [str(p) for p in required if not p.exists()]
assert not missing, "Missing required paths:\n" + "\n".join(missing)

run_dirs = sorted([p for p in RUN_DIR.iterdir() if p.is_dir() and (p / "config.yml").is_file()])
assert len(run_dirs) == 80, f"Expected 80 valid run folders, found {len(run_dirs)}"

run_re = re.compile(r"Top18_Add8_rank(?P<rank>\d+)_(?P<token>\d{4}_\d{6})_SE(?P<epoch>\d+)_RS(?P<seed>\d+)_")
rows=[]
for rd in run_dirs:
    m=run_re.search(rd.name)
    assert m, f"Cannot parse {rd.name}"
    d=m.groupdict(); epoch=int(d['epoch'])
    test_p=rd / 'test' / f"model_epoch{epoch:03d}" / 'test_results.p'
    rows.append({'run_dir':rd,'name':rd.name,'rank':int(d['rank']),'token':d['token'],
                 'epoch':epoch,'seed':d['seed'],'test_results':test_p,'test_ok':test_p.is_file()})
runs_df=pd.DataFrame(rows)
assert len(runs_df)==80 and runs_df['test_ok'].all()
assert runs_df['rank'].nunique()==8 and runs_df['seed'].nunique()==10
print('80 TEST RUNS: PASS')
display(runs_df.groupby('rank').agg(n_runs=('name','size'), n_seeds=('seed','nunique')))


80 TEST RUNS: PASS


,n_runs,n_seeds
rank,,
10,10,10
11,10,10
13,10,10
14,10,10
15,10,10
16,10,10
17,10,10
18,10,10


In [3]:
obs = read_pickle_compat(OBS_P)
ens_top10 = read_pickle_compat(ENS_TOP10_P)
old_metrics_overall = read_pickle_compat(OLD_METRICS_OVERALL_P)
assert isinstance(obs,dict) and len(obs)==531
assert isinstance(ens_top10,dict) and len(ens_top10)==531
assert isinstance(old_metrics_overall,dict) and len(old_metrics_overall)==531

BASINS=list(obs.keys())
SEEDS=list(next(iter(ens_top10.values())).keys())
sample_basin=BASINS[0]; sample_seed=SEEDS[0]
obs0=obs[sample_basin]
pred10_0=ens_top10[sample_basin][sample_seed]['ensemble']
DATES=pd.DatetimeIndex(obs0['date'].values)
N_TIME=len(DATES)
assert len(BASINS)==531 and len(SEEDS)==10 and N_TIME==3652
assert DATES.min()==pd.Timestamp('1989-10-01') and DATES.max()==pd.Timestamp('1999-09-30')
assert set(runs_df['seed'])==set(SEEDS)
assert str(pred10_0.attrs.get('ensemble_agg')).lower()=='median'
assert int(pred10_0.attrs.get('n_members'))==10
print('CANONICAL SCHEMA: PASS')
print('Seeds:',SEEDS)
print('Dates:',DATES.min().date(),'to',DATES.max().date())


CANONICAL SCHEMA: PASS
Seeds: ['328237', '985957', '473461', '993061', '307443', '385445', '845563', '643969', '665980', '191889']
Dates: 1989-10-01 to 1999-09-30


In [4]:
def canonical_obs_da(basin):
    da=obs[basin]
    if 'time_step' not in da.dims:
        da=da.expand_dims(time_step=[0], axis=-1)
    return da.transpose('date','time_step').sel(date=DATES)

def standard_mape(obs_da, sim_da, mode='positive'):
    o=np.asarray(obs_da.values,dtype=float).reshape(-1)
    s=np.asarray(sim_da.values,dtype=float).reshape(-1)
    valid=np.isfinite(o)&np.isfinite(s)
    if mode=='positive':
        valid &= o>0; denom=o[valid]
    elif mode=='nonzero':
        valid &= o!=0; denom=np.abs(o[valid])
    elif mode=='epsilon':
        denom=np.maximum(np.abs(o[valid]),np.finfo(float).eps)
    else: raise ValueError(mode)
    if valid.sum()==0: return np.nan
    return float(np.mean(np.abs((s[valid]-o[valid])/denom))*100)

NH_METRIC_LIST=['NSE','MSE','RMSE','KGE','Alpha-NSE','Pearson-r','Beta-KGE','Beta-NSE',
                'FHV','FMS','FLV','Peak-Timing','Peak-MAPE','Missed-Peaks']

def calc_metrics(obs_da, sim_da, do_clip=True, mape_mode='positive'):
    # NeuralHydrology metrics in the installed local version expect
    # a 1-D DataArray indexed by date. Our stored artifacts have the
    # canonical shape (date, time_step=1), so remove ONLY that singleton
    # dimension before calling NH metrics.
    o,s=xr.align(obs_da,sim_da,join='inner')

    if 'time_step' in o.dims:
        if o.sizes['time_step'] != 1:
            raise ValueError(f"Observation time_step must be singleton, got {o.sizes['time_step']}")
        o=o.isel(time_step=0,drop=True)

    if 'time_step' in s.dims:
        if s.sizes['time_step'] != 1:
            raise ValueError(f"Simulation time_step must be singleton, got {s.sizes['time_step']}")
        s=s.isel(time_step=0,drop=True)

    # Final alignment after squeezing keeps the common date coordinate exact.
    o,s=xr.align(o,s,join='inner')

    if o.ndim != 1 or s.ndim != 1:
        raise ValueError(f"NH metric inputs must be 1-D after squeeze; obs={o.dims}, sim={s.dims}")

    if do_clip:
        s=s.clip(min=0)

    try:
        d=nh_metrics.calculate_metrics(
            o,s,
            metrics=NH_METRIC_LIST,
            resolution='1D',
            datetime_coord='date'
        )
    except TypeError:
        d=nh_metrics.calculate_metrics(
            o,s,
            metrics=NH_METRIC_LIST,
            resolution='1D'
        )

    d['MAPE']=standard_mape(o,s,mode=mape_mode)
    return {k:float(v) if np.isscalar(v) else v for k,v in d.items()}

def extract_sim_da(result_obj, basin):
    node=result_obj[basin]
    if isinstance(node,dict) and '1D' in node: node=node['1D']
    if isinstance(node,dict) and 'xr' in node: node=node['xr']
    if isinstance(node,xr.Dataset):
        vars_=[v for v in node.data_vars if '_sim' in str(v).lower()]
        if not vars_: vars_=[v for v in node.data_vars if 'sim' in str(v).lower()]
        assert vars_, f"No sim variable in {list(node.data_vars)}"
        da=node[vars_[0]]
    elif isinstance(node,xr.DataArray): da=node
    else: raise TypeError(type(node))
    if 'time_step' not in da.dims: da=da.expand_dims(time_step=[0],axis=-1)
    return da.transpose('date','time_step').sel(date=DATES)

def flatten_metric_tree(obj, known_basins=None, known_seeds=None):
    known_basins=set(known_basins or []); known_seeds=set(map(str,known_seeds or [])); records=[]
    def walk(x,path=()):
        if isinstance(x,dict):
            metric_keys=[k for k in x if k in NH_METRIC_LIST or k=='MAPE']
            if metric_keys:
                toks=[str(t) for t in path]
                rec={'path':'|'.join(toks)}
                for k in metric_keys: rec[k]=x.get(k,np.nan)
                rec['basin']=next((t for t in toks if t in known_basins),None)
                rec['seed']=next((t for t in toks if t in known_seeds),None)
                rec['method']=next((t for t in toks if t in ['Top10','Cluster-wise','Benchmark','Top18','Top18-mean']),None)
                rec['year']=next((int(t) for t in toks if re.fullmatch(r'(19|20)\d{2}',t)),None)
                rec['season']=next((t for t in toks if t in ['JFM','AMJ','JAS','OND','DJF','MAM','JJA','SON','Winter','Spring','Summer','Autumn','Fall']),None)
                records.append(rec)
            else:
                for k,v in x.items(): walk(v,path+(k,))
        elif isinstance(x,(list,tuple)):
            for i,v in enumerate(x): walk(v,path+(i,))
    walk(obj)
    return pd.DataFrame(records)

def save_df(df,name):
    p=OUT_TABLES/name; df.to_csv(p,index=False); return p

def figure_save(fig,stem):
    png=OUT_FIGURES/f'{stem}.png'; pdf=OUT_FIGURES/f'{stem}.pdf'
    fig.savefig(png,dpi=220,bbox_inches='tight'); fig.savefig(pdf,bbox_inches='tight'); plt.close(fig)
    return png,pdf

print('Helpers ready.')


Helpers ready.


In [5]:
ADD8_NPY=OUT_CACHE/'add8_predictions.npy'
ADD8_META=OUT_CACHE/'add8_metadata.json'
RANKS=sorted(runs_df['rank'].unique())
seed_to_i={s:i for i,s in enumerate(SEEDS)}; rank_to_i={int(r):i for i,r in enumerate(RANKS)}; basin_to_i={b:i for i,b in enumerate(BASINS)}
expected_shape=(len(SEEDS),len(RANKS),len(BASINS),N_TIME)

if ADD8_NPY.exists() and ADD8_META.exists():
    meta=json.loads(ADD8_META.read_text(encoding='utf-8'))
    if tuple(meta['shape'])!=expected_shape:
        ADD8_NPY.unlink(); ADD8_META.unlink()

if not ADD8_NPY.exists():
    mm=np.lib.format.open_memmap(ADD8_NPY,mode='w+',dtype=np.float32,shape=expected_shape); mm[:]=np.nan
    for n,row in runs_df.sort_values(['rank','seed']).reset_index(drop=True).iterrows():
        print(f"[{n+1:02d}/80] rank={row['rank']} seed={row['seed']}")
        result=read_pickle_compat(Path(row['test_results']))
        si=seed_to_i[row['seed']]; ri=rank_to_i[int(row['rank'])]
        for b in BASINS:
            mm[si,ri,basin_to_i[b],:]=np.asarray(extract_sim_da(result,b).values,dtype=np.float32).reshape(-1)
        del result; gc.collect()
    mm.flush(); del mm
    ADD8_META.write_text(json.dumps({'shape':list(expected_shape),'axes':['seed','rank','basin','date'],
        'seeds':SEEDS,'ranks':[int(x) for x in RANKS],'basins':BASINS,
        'dates':[str(d.date()) for d in DATES],'source':str(RUN_DIR)},indent=2),encoding='utf-8')

add8=np.load(ADD8_NPY,mmap_mode='r')
assert add8.shape==expected_shape
print('ADD8 MEMMAP: PASS',add8.shape)


ADD8 MEMMAP: PASS (10, 8, 531, 3652)


In [6]:
# ============================================================
# 5. CONVERT LEGACY All_Top10 PICKLE -> COMPACT MEMMAP
#
# Robust to legacy nesting:
#   - basin does NOT need to be a top-level key
#   - basin IDs may be int / string / lack the leading zero
#   - seed may be a key or embedded as RSxxxxxx in a run/config key
#
# The converter walks all simulation DataArrays once and requires
# exactly 531 basins x 10 seeds x 10 Top10 members = 53,100 leaves.
# It runs in a child process so the ~3.7 GB pickle is released when
# conversion finishes.
# ============================================================

TOP10_NPY = OUT_CACHE / "top10_raw_predictions.npy"
TOP10_META = OUT_CACHE / "top10_raw_metadata.json"
HELPER = OUT_CACHE / "_convert_legacy_top10.py"
HELPER_LOG = OUT_AUDIT / "convert_legacy_top10.log"

helper_code = r"""
from pathlib import Path
import sys, json, re, gc
import numpy as np
import pandas as pd
import xarray as xr

raw_p = Path(sys.argv[1])
ens_p = Path(sys.argv[2])
obs_p = Path(sys.argv[3])
out_npy = Path(sys.argv[4])
out_json = Path(sys.argv[5])

def readp(p):
    return pd.read_pickle(p)

def iter_sim_leaves(obj, path=()):
    if isinstance(obj, xr.Dataset):
        sim_vars = [v for v in obj.data_vars if "_sim" in str(v).lower()]
        if not sim_vars:
            sim_vars = [v for v in obj.data_vars if "sim" in str(v).lower()]
        for v in sim_vars:
            yield path + (str(v),), obj[v]
        return

    if isinstance(obj, xr.DataArray):
        nm = str(obj.name or "").lower()
        path_text = "|".join(map(str, path)).lower()
        if ("_sim" in nm) or ("sim" in nm) or ("_sim" in path_text):
            yield path, obj
        return

    if isinstance(obj, dict):
        for k, v in obj.items():
            yield from iter_sim_leaves(v, path + (k,))
        return

    if isinstance(obj, (list, tuple)):
        for i, v in enumerate(obj):
            yield from iter_sim_leaves(v, path + (i,))
        return

def canonical_basin_token(token, basin_set):
    s = str(token)

    if s in basin_set:
        return s

    if re.fullmatch(r"\d{7,8}", s):
        z = s.zfill(8)
        if z in basin_set:
            return z

    for m in re.findall(r"(?<!\d)(\d{7,8})(?!\d)", s):
        z = m.zfill(8)
        if z in basin_set:
            return z

    return None

def seed_from_path(path, seeds):
    toks = [str(x) for x in path]

    for s in seeds:
        if s in toks:
            return s

    joined = "|".join(toks)
    for s in seeds:
        if f"RS{s}" in joined:
            return s
        if re.search(rf"(?<!\d){re.escape(s)}(?!\d)", joined):
            return s

    return None

def basin_from_path(path, basin_set):
    hits = []
    for token in path:
        b = canonical_basin_token(token, basin_set)
        if b is not None:
            hits.append(b)

    hits = list(dict.fromkeys(hits))

    if len(hits) == 1:
        return hits[0]

    if len(hits) > 1:
        raise RuntimeError(f"Ambiguous basin IDs in path: {path} -> {hits}")

    return None

def member_label_from_path(path, basin, seed, basin_set):
    reduced = []

    generic_exact = {
        "1D", "xr", "ensemble", "sim", "simulation",
        "prediction", "predictions", "QObs(mm/d)_sim",
        "QObs(mm/d)_obs"
    }

    for token in path:
        t = str(token)

        if canonical_basin_token(t, basin_set) == basin:
            continue

        if t == seed:
            continue

        if t in generic_exact or "_sim" in t.lower():
            continue

        t = re.sub(rf"_?RS{re.escape(seed)}", "", t)
        t = re.sub(rf"(?<!\d){re.escape(seed)}(?!\d)", "", t)
        t = re.sub(r"__+", "_", t).strip("_| ")

        if t:
            reduced.append(t)

    label = "|".join(reduced)
    return label if label else "member"

obs = readp(obs_p)
ens = readp(ens_p)

basins = list(obs.keys())
basin_set = set(basins)
basin_i = {b: i for i, b in enumerate(basins)}

seeds = list(next(iter(ens.values())).keys())
seed_i = {s: i for i, s in enumerate(seeds)}

dates = pd.DatetimeIndex(obs[basins[0]]["date"].values)
n_time = len(dates)

if len(basins) != 531:
    raise RuntimeError(f"Expected 531 canonical basins, found {len(basins)}")

if len(seeds) != 10:
    raise RuntimeError(f"Expected 10 canonical seeds, found {len(seeds)}")

if n_time != 3652:
    raise RuntimeError(f"Expected 3652 test days, found {n_time}")

print("Canonical basins:", len(basins), flush=True)
print("Canonical seeds :", seeds, flush=True)
print("Canonical dates :", dates.min(), "to", dates.max(), flush=True)

print("Loading legacy raw Top10:", raw_p, flush=True)
raw = readp(raw_p)
print("Loaded raw object type:", type(raw), flush=True)

if isinstance(raw, dict):
    print("Legacy top-level key sample:", list(raw.keys())[:12], flush=True)

shape = (len(seeds), 10, len(basins), n_time)

if out_npy.exists():
    out_npy.unlink()

if out_json.exists():
    out_json.unlink()

mm = np.lib.format.open_memmap(
    out_npy, mode="w+", dtype=np.float32, shape=shape
)
mm[:] = np.nan

member_to_i = {s: {} for s in seeds}
assigned = np.zeros((len(seeds), 10, len(basins)), dtype=bool)

n_sim_leaves = 0
n_written = 0
n_no_basin = 0
n_no_seed = 0
debug_paths = []
unmatched_examples = []

for path, da in iter_sim_leaves(raw):
    n_sim_leaves += 1

    if len(debug_paths) < 30:
        debug_paths.append(tuple(map(str, path)))

    basin = basin_from_path(path, basin_set)
    seed = seed_from_path(path, seeds)

    if basin is None:
        n_no_basin += 1
        if len(unmatched_examples) < 30:
            unmatched_examples.append(("NO_BASIN", tuple(map(str, path))))
        continue

    if seed is None:
        n_no_seed += 1
        if len(unmatched_examples) < 30:
            unmatched_examples.append(("NO_SEED", tuple(map(str, path))))
        continue

    label = member_label_from_path(path, basin, seed, basin_set)
    mapping = member_to_i[seed]

    if label not in mapping:
        if len(mapping) >= 10:
            raise RuntimeError(
                f"More than 10 logical members discovered for seed={seed}. "
                f"New label={label!r}; existing={list(mapping)}; path={path}"
            )
        mapping[label] = len(mapping)
        print(
            f"Discovered member seed={seed} index={mapping[label]} label={label}",
            flush=True
        )

    mi = mapping[label]
    si = seed_i[seed]
    bi = basin_i[basin]

    if assigned[si, mi, bi]:
        raise RuntimeError(
            f"Duplicate simulation leaf for basin={basin} seed={seed} "
            f"member={label!r}; path={path}"
        )

    if "date" in da.coords:
        try:
            da = da.sel(date=dates)
        except Exception as e:
            raise RuntimeError(
                f"Date alignment failed for basin={basin} seed={seed} "
                f"member={label!r}: {e}"
            )

    arr = np.asarray(da.values, dtype=np.float32).reshape(-1)

    if len(arr) != n_time:
        raise RuntimeError(
            f"Length mismatch basin={basin} seed={seed} member={label!r}: "
            f"{len(arr)} vs expected {n_time}; path={path}"
        )

    mm[si, mi, bi, :] = arr
    assigned[si, mi, bi] = True
    n_written += 1

    if n_written % 5000 == 0:
        print(f"Written {n_written}/53100 expected predictions", flush=True)

print("Simulation leaves seen :", n_sim_leaves, flush=True)
print("Predictions written    :", n_written, flush=True)
print("Leaves without basin   :", n_no_basin, flush=True)
print("Leaves without seed    :", n_no_seed, flush=True)
print(
    "Members per seed       :",
    {s: len(v) for s, v in member_to_i.items()},
    flush=True
)

expected = len(basins) * len(seeds) * 10
missing_assignments = int((~assigned).sum())

if n_written != expected or missing_assignments != 0:
    print("FIRST SIM PATHS:", flush=True)
    for p in debug_paths:
        print("  ", p, flush=True)

    print("UNMATCHED EXAMPLES:", flush=True)
    for reason, p in unmatched_examples:
        print("  ", reason, p, flush=True)

    for s in seeds:
        si = seed_i[s]
        print(
            "COVERAGE",
            s,
            "members=", len(member_to_i[s]),
            "assigned=", int(assigned[si].sum()),
            "expected=", 10 * len(basins),
            flush=True
        )

    raise RuntimeError(
        f"Legacy Top10 conversion incomplete: wrote {n_written}/{expected}; "
        f"missing assignment cells={missing_assignments}"
    )

if not all(len(member_to_i[s]) == 10 for s in seeds):
    raise RuntimeError(
        f"Expected exactly 10 member labels per seed: "
        f"{ {s: len(v) for s,v in member_to_i.items()} }"
    )

member_labels_by_seed = {}

for s in seeds:
    inv = sorted(member_to_i[s].items(), key=lambda kv: kv[1])
    member_labels_by_seed[s] = [label for label, _ in inv]

mm.flush()
del mm
gc.collect()

meta = {
    "shape": list(shape),
    "axes": ["seed", "member", "basin", "date"],
    "seeds": seeds,
    "basins": basins,
    "dates": [str(d.date()) for d in dates],
    "member_labels_by_seed": member_labels_by_seed,
    "source": str(raw_p),
    "expected_predictions": expected,
    "written_predictions": n_written,
}

out_json.write_text(json.dumps(meta, indent=2), encoding="utf-8")

print("TOP10 LEGACY CONVERSION: PASS", flush=True)
print("Shape:", shape, flush=True)
print("DONE", flush=True)
"""

HELPER.write_text(helper_code, encoding="utf-8")

need = True

if TOP10_NPY.exists() and TOP10_META.exists():
    try:
        meta10 = json.loads(TOP10_META.read_text(encoding="utf-8"))
        arr = np.load(TOP10_NPY, mmap_mode="r")

        if (
            tuple(meta10.get("shape", [])) == (10, 10, 531, 3652)
            and arr.shape == (10, 10, 531, 3652)
            and meta10.get("seeds") == SEEDS
            and meta10.get("basins") == BASINS
            and int(meta10.get("written_predictions", 53100)) == 53100
        ):
            need = False
            print("Reusing validated Top10 compact cache:", TOP10_NPY)

        del arr

    except Exception:
        need = True

if need:
    print("Converting 3.7 GB legacy Top10 in child process.")
    print("Log:", HELPER_LOG)

    with HELPER_LOG.open("w", encoding="utf-8") as log:
        proc = subprocess.run(
            [
                sys.executable,
                str(HELPER),
                str(RAW_TOP10_P),
                str(ENS_TOP10_P),
                str(OBS_P),
                str(TOP10_NPY),
                str(TOP10_META),
            ],
            stdout=log,
            stderr=subprocess.STDOUT,
        )

    log_text = HELPER_LOG.read_text(encoding="utf-8", errors="replace")

    if proc.returncode != 0:
        print(log_text[-20000:])
        raise RuntimeError(
            "Top10 conversion failed. The diagnostic tail is printed above "
            "and the full log is in audit/convert_legacy_top10.log"
        )

    print(log_text[-5000:])

top10_raw = np.load(TOP10_NPY, mmap_mode="r")
meta10 = json.loads(TOP10_META.read_text(encoding="utf-8"))

assert top10_raw.shape == (10, 10, 531, 3652)
assert meta10["seeds"] == SEEDS
assert meta10["basins"] == BASINS
assert int(meta10.get("written_predictions", 53100)) == 53100

print("TOP10 MEMMAP: PASS", top10_raw.shape)


Reusing validated Top10 compact cache: F:\Experiments\CAMELS_US\Clean_4_upload\Revision_2026_Top18_4Framework\Top18_vs_Top10_Final_Analysis\cache\top10_raw_predictions.npy
TOP10 MEMMAP: PASS (10, 10, 531, 3652)


In [7]:
RAW_ADD8_P=OUT_PICKLES/'All_Add8_Configs_CAMELSUS_test_compact.p'
RAW_TOP18_P=OUT_PICKLES/'All_Top18_Configs_CAMELSUS_test.p'
ENS_TOP18_P=OUT_PICKLES/'ensemble_Top18_CAMELSUS_test.p'
ENS_TOP18_MEAN_P=OUT_PICKLES/'ensemble_Top18_mean_CAMELSUS_test.p'
labels10=meta10['member_labels_by_seed']; add8_labels=[f'rank{r}' for r in RANKS]

def member_stack(arr,labels,attrs):
    return xr.DataArray(arr[:,:,None].astype(np.float32,copy=False),dims=('member','date','time_step'),coords={'member':labels,'date':DATES,'time_step':[0]},name='QObs(mm/d)_sim',attrs=attrs)
def ensemble_da(arr,agg):
    v=np.nanmedian(arr,axis=0) if agg=='median' else np.nanmean(arr,axis=0)
    return xr.DataArray(v[:,None].astype(np.float32),dims=('date','time_step'),coords={'date':DATES,'time_step':[0]},name='Ensemble(mm/d)_sim',attrs={'ensemble_agg':agg,'n_members':int(arr.shape[0])})
raw_add8={} if WRITE_COMPACT_RAW_ADD8 else None; raw18={} if WRITE_COMPACT_RAW_TOP18 else None; ens18={}; ens18_mean={}
for bi,b in enumerate(BASINS):
    ens18[b]={}; ens18_mean[b]={}
    if raw_add8 is not None: raw_add8[b]={}
    if raw18 is not None: raw18[b]={}
    for si,s in enumerate(SEEDS):
        a8=np.asarray(add8[si,:,bi,:],dtype=np.float32); t10=np.asarray(top10_raw[si,:,bi,:],dtype=np.float32); all18=np.concatenate([t10,a8],axis=0)
        ens18[b][s]={'ensemble':ensemble_da(all18,'median')}; ens18_mean[b][s]={'ensemble':ensemble_da(all18,'mean')}
        if raw_add8 is not None: raw_add8[b][s]={'members':member_stack(a8,add8_labels,{'n_members':8,'member_family':'Top18_Add8','schema':'compact_member_stack_v1'})}
        if raw18 is not None:
            labs=[f'Top10::{x}' for x in labels10[s]]+[f'Add8::{x}' for x in add8_labels]
            raw18[b][s]={'members':member_stack(all18,labs,{'n_members':18,'aggregation_primary':'median','schema':'compact_member_stack_v1'})}
    if (bi+1)%50==0 or bi==len(BASINS)-1: print(f'Built {bi+1}/{len(BASINS)} basins')
pd.to_pickle(ens18,ENS_TOP18_P); pd.to_pickle(ens18_mean,ENS_TOP18_MEAN_P)
if raw_add8 is not None: pd.to_pickle(raw_add8,RAW_ADD8_P); del raw_add8
if raw18 is not None: pd.to_pickle(raw18,RAW_TOP18_P); del raw18
gc.collect()
assert len(pd.read_pickle(ENS_TOP18_P))==531
print('TOP18 ARTIFACTS: PASS')
print(ENS_TOP18_P); print(ENS_TOP18_MEAN_P); print(RAW_TOP18_P)


Built 50/531 basins
Built 100/531 basins
Built 150/531 basins
Built 200/531 basins
Built 250/531 basins
Built 300/531 basins
Built 350/531 basins
Built 400/531 basins
Built 450/531 basins
Built 500/531 basins
Built 531/531 basins
TOP18 ARTIFACTS: PASS
F:\Experiments\CAMELS_US\Clean_4_upload\Revision_2026_Top18_4Framework\Top18_vs_Top10_Final_Analysis\pickles\ensemble_Top18_CAMELSUS_test.p
F:\Experiments\CAMELS_US\Clean_4_upload\Revision_2026_Top18_4Framework\Top18_vs_Top10_Final_Analysis\pickles\ensemble_Top18_mean_CAMELSUS_test.p
F:\Experiments\CAMELS_US\Clean_4_upload\Revision_2026_Top18_4Framework\Top18_vs_Top10_Final_Analysis\pickles\All_Top18_Configs_CAMELSUS_test.p


In [8]:
def old_leaf(b,s,method='Top10'): return old_metrics_overall[b][s][method]
samples=[(BASINS[i],SEEDS[j]) for i,j in [(0,0),(5,1),(50,2),(150,5),(400,8)]]
rows=[]
for clip in [True,False]:
    for mmode in ['positive','nonzero','epsilon']:
        for b,s in samples:
            old=old_leaf(b,s); new=calc_metrics(canonical_obs_da(b),ens_top10[b][s]['ensemble'],do_clip=clip,mape_mode=mmode)
            rec={'basin':b,'seed':s,'clip':clip,'mape_mode':mmode}
            for k in NH_METRIC_LIST+['MAPE']: rec[f'absdiff_{k}']=abs(float(new.get(k,np.nan))-float(old.get(k,np.nan)))
            rows.append(rec)
audit=pd.DataFrame(rows); nhcols=[f'absdiff_{k}' for k in NH_METRIC_LIST]
audit['nh_mean_absdiff']=audit[nhcols].mean(axis=1,skipna=True); audit['mape_absdiff']=audit['absdiff_MAPE']
summary=audit.groupby(['clip','mape_mode'],as_index=False).agg(nh_mean_absdiff=('nh_mean_absdiff','mean'),nh_max_absdiff=('nh_mean_absdiff','max'),mape_mean_absdiff=('mape_absdiff','mean'),mape_max_absdiff=('mape_absdiff','max')).sort_values(['nh_mean_absdiff','mape_mean_absdiff'])
save_df(audit,'metric_protocol_audit_samples.csv'); save_df(summary,'metric_protocol_candidates.csv'); display(summary)
DO_CLIP=bool(summary.iloc[0]['clip']); msub=summary[summary['clip']==DO_CLIP].sort_values('mape_mean_absdiff'); MAPE_MODE=str(msub.iloc[0]['mape_mode'])
print('Selected:',DO_CLIP,MAPE_MODE)
assert float(summary.iloc[0]['nh_mean_absdiff'])<=5e-4, 'NH metric protocol mismatch; inspect audit tables.'
assert float(msub.iloc[0]['mape_mean_absdiff'])<=1e-2, 'MAPE protocol mismatch; inspect audit tables.'
print('METRIC PROTOCOL: PASS')


,clip,mape_mode,nh_mean_absdiff,nh_max_absdiff,mape_mean_absdiff,mape_max_absdiff
0,False,epsilon,0.0,0.0,8.104177e-07,0.000002
1,False,nonzero,0.0,0.0,8.104177e-07,0.000002
2,False,positive,0.0,0.0,8.104177e-07,0.000002
3,True,epsilon,0.0,0.0,8.104177e-07,0.000002
4,True,nonzero,0.0,0.0,8.104177e-07,0.000002
5,True,positive,0.0,0.0,8.104177e-07,0.000002


Selected: False epsilon
METRIC PROTOCOL: PASS


In [9]:
METRICS_TOP18_ONLY_P=OUT_PICKLES/'metrics_all_test_period_top18_only.p'; METRICS_WITH_TOP18_P=OUT_PICKLES/'metrics_all_test_period_with_top18.p'
metrics18={}; rows=[]
for bi,b in enumerate(BASINS):
    metrics18[b]={}; o=canonical_obs_da(b)
    for s in SEEDS:
        metrics18[b][s]={}
        m=calc_metrics(o,ens18[b][s]['ensemble'],DO_CLIP,MAPE_MODE); metrics18[b][s]['Top18']=m
        for k,v in m.items(): rows.append({'basin':b,'seed':s,'method':'Top18','metric':k,'value':v})
        if CALCULATE_TOP18_MEAN_OVERALL:
            mm=calc_metrics(o,ens18_mean[b][s]['ensemble'],DO_CLIP,MAPE_MODE); metrics18[b][s]['Top18-mean']=mm
            for k,v in mm.items(): rows.append({'basin':b,'seed':s,'method':'Top18-mean','metric':k,'value':v})
    if (bi+1)%50==0 or bi==len(BASINS)-1: print(f'Overall {bi+1}/{len(BASINS)}')
pd.to_pickle(metrics18,METRICS_TOP18_ONLY_P)
combined={b:{s:{**old_metrics_overall[b][s],**metrics18[b][s]} for s in SEEDS} for b in BASINS}; pd.to_pickle(combined,METRICS_WITH_TOP18_P)
for b in BASINS:
    for s in SEEDS:
        for method,md in old_metrics_overall[b][s].items():
            for k,v in md.items(): rows.append({'basin':b,'seed':s,'method':method,'metric':k,'value':v})
overall_long=pd.DataFrame(rows); save_df(overall_long,'overall_metrics_long.csv')
basin_method=overall_long.groupby(['basin','method','metric'],as_index=False)['value'].median().rename(columns={'value':'basin_seed_median'}); save_df(basin_method,'overall_basin_seed_medians.csv')
summary_overall=basin_method.groupby(['method','metric'])['basin_seed_median'].agg(['count','median','mean','std','min','max']).reset_index(); save_df(summary_overall,'overall_summary_by_method_metric.csv')
print('OVERALL METRICS: PASS'); display(summary_overall[summary_overall['metric'].isin(['NSE','KGE','RMSE'])])


Overall 50/531
Overall 100/531
Overall 150/531
Overall 200/531
Overall 250/531
Overall 300/531
Overall 350/531
Overall 400/531
Overall 450/531
Overall 500/531
Overall 531/531
OVERALL METRICS: PASS


,method,metric,count,median,mean,std,min,max
6,Benchmark,KGE,531,0.790749,0.736999,0.212666,-2.265997,0.952060
10,Benchmark,NSE,531,0.793116,0.755212,0.146424,-0.355723,0.948496
14,Benchmark,RMSE,531,1.120154,1.314555,0.856498,0.066941,5.649752
21,Cluster-wise,KGE,531,0.786199,0.732776,0.199584,-1.509731,0.966628
25,Cluster-wise,NSE,531,0.815791,0.776305,0.144630,-0.307649,0.963562
29,Cluster-wise,RMSE,531,1.062430,1.247052,0.817096,0.065741,5.598154
36,Top10,KGE,531,0.771335,0.728102,0.201153,-1.191167,0.961606
40,Top10,NSE,531,0.811914,0.769512,0.161461,-0.830943,0.963863
44,Top10,RMSE,531,1.062036,1.246528,0.803306,0.077784,5.449623
51,Top18,KGE,531,0.774500,0.729393,0.196575,-1.028689,0.960132


In [10]:
primary=basin_method[basin_method['method']=='Top18'].rename(columns={'basin_seed_median':'top18'})[['basin','metric','top18']]
out=[]
for baseline in ['Top10','Cluster-wise','Benchmark']:
    base=basin_method[basin_method['method']==baseline].rename(columns={'basin_seed_median':'baseline'})[['basin','metric','baseline']]
    m=primary.merge(base,on=['basin','metric'])
    for metric,g in m.groupby('metric'):
        x=g['top18'].to_numpy(float); y=g['baseline'].to_numpy(float); valid=np.isfinite(x)&np.isfinite(y)
        if metric in ['NSE','KGE','Pearson-r']: improved=x>y
        elif metric in ['MSE','RMSE','Peak-Timing','Peak-MAPE','Missed-Peaks','MAPE']: improved=x<y
        elif metric in ['FHV','FMS','FLV','Beta-NSE']: improved=np.abs(x)<np.abs(y)
        else: improved=x>y
        try: stat,p=wilcoxon(x[valid],y[valid],zero_method='wilcox')
        except Exception: stat,p=np.nan,np.nan
        out.append({'comparison':f'Top18 vs {baseline}','baseline':baseline,'metric':metric,'n_basins':int(valid.sum()),
                    'median_top18':float(np.nanmedian(x)),'median_baseline':float(np.nanmedian(y)),
                    'median_raw_delta_top18_minus_baseline':float(np.nanmedian(x-y)),
                    'mean_raw_delta_top18_minus_baseline':float(np.nanmean(x-y)),
                    'pct_basins_improved':float(np.nanmean(improved[valid])*100),'wilcoxon_stat':stat,'wilcoxon_p':p})
paired_summary=pd.DataFrame(out); save_df(paired_summary,'paired_comparison_summary.csv'); display(paired_summary[paired_summary['metric'].isin(KEY_METRICS)])


,comparison,baseline,metric,n_basins,median_top18,median_baseline,median_raw_delta_top18_minus_baseline,mean_raw_delta_top18_minus_baseline,pct_basins_improved,wilcoxon_stat,wilcoxon_p
3,Top18 vs Top10,Top10,FHV,531,-16.962702,-16.374217,-0.289125,-2.387030e-01,34.651601,44875.0,3.359824e-13
4,Top18 vs Top10,Top10,FLV,531,-2.010580,-2.922845,-0.219944,-3.256609e+09,42.937853,62506.0,2.601754e-02
5,Top18 vs Top10,Top10,FMS,531,-7.665345,-6.935979,-0.657554,6.750155e+02,54.237288,32311.0,2.453142e-27
6,Top18 vs Top10,Top10,KGE,531,0.774500,0.771335,0.000044,1.290610e-03,50.282486,68904.0,6.269866e-01
7,Top18 vs Top10,Top10,MAPE,531,34.083099,34.721059,-0.559687,3.563770e+14,70.433145,58494.0,6.059255e-04
9,Top18 vs Top10,Top10,Missed-Peaks,531,0.330000,0.333333,0.000000,-2.005353e-03,34.839925,27214.5,5.347989e-02
10,Top18 vs Top10,Top10,NSE,531,0.814496,0.811914,0.002059,2.310765e-03,76.271186,29135.0,9.059616e-32
11,Top18 vs Top10,Top10,Peak-MAPE,531,34.376804,34.907177,-0.142086,-1.760010e-01,59.887006,52381.0,2.507592e-07
12,Top18 vs Top10,Top10,Peak-Timing,531,0.238095,0.236842,0.000000,-3.465539e-03,24.858757,11596.5,1.774041e-02
14,Top18 vs Top10,Top10,RMSE,531,1.056349,1.062036,-0.006114,-7.395751e-03,76.271186,25800.0,8.479373e-37


In [11]:
WY_P=OUT_PICKLES/'metrics_by_water_year_top18.p'; wy_rows=[]; wy_pickle={}
for bi,b in enumerate(BASINS):
    wy_pickle[b]={}; o_full=canonical_obs_da(b)
    for s in SEEDS:
        wy_pickle[b][s]={}; p_full=ens18[b][s]['ensemble']
        for wy in range(1990,2000):
            o=o_full.sel(date=slice(pd.Timestamp(wy-1,10,1),pd.Timestamp(wy,9,30))); p=p_full.sel(date=o['date'])
            md=calc_metrics(o,p,DO_CLIP,MAPE_MODE); wy_pickle[b][s][str(wy)]={'Top18':md}
            for k,v in md.items(): wy_rows.append({'basin':b,'seed':s,'year':wy,'method':'Top18','metric':k,'value':v})
    if (bi+1)%50==0 or bi==len(BASINS)-1: print(f'WY {bi+1}/{len(BASINS)}')
pd.to_pickle(wy_pickle,WY_P); wy_top18_long=pd.DataFrame(wy_rows); save_df(wy_top18_long,'water_year_top18_long.csv')
old_wy=read_pickle_compat(OLD_METRICS_WY_P); flat=flatten_metric_tree(old_wy,BASINS,SEEDS); save_df(flat,'water_year_old_flat_audit.csv')
oldrows=[]
for _,r in flat.iterrows():
    if r.get('method') is None or pd.isna(r.get('year')): continue
    for metric in NH_METRIC_LIST+['MAPE']:
        if metric in r and pd.notna(r[metric]): oldrows.append({'basin':r['basin'],'seed':r['seed'],'year':int(r['year']),'method':r['method'],'metric':metric,'value':r[metric]})
wy_combined=pd.concat([pd.DataFrame(oldrows),wy_top18_long],ignore_index=True); save_df(wy_combined,'water_year_comparison_long.csv'); del old_wy; gc.collect(); print('WATER-YEAR: PASS')


WY 50/531
WY 100/531
WY 150/531
WY 200/531
WY 250/531
WY 300/531
WY 350/531


C:\Users\hosseinifh\neuralhydrology\neuralhydrology\evaluation\metrics.py:310: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  r, _ = stats.pearsonr(obs.values, sim.values)
C:\Users\hosseinifh\neuralhydrology\neuralhydrology\evaluation\metrics.py:346: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  r, _ = stats.pearsonr(obs.values, sim.values)
C:\Users\hosseinifh\neuralhydrology\neuralhydrology\evaluation\metrics.py:467: RuntimeWarning: divide by zero encountered in scalar divide
  fhv = np.sum(sim - obs) / np.sum(obs)
C:\Users\hosseinifh\neuralhydrology\neuralhydrology\evaluation\metrics.py:310: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  r, _ = stats.pearsonr(obs.values, sim.values)
C:\Users\hosseinifh\neuralhydrology\neuralhydrology\evaluation\metrics.py:346: ConstantInputWarning: An input array is constant; the correlation coefficient i

WY 400/531
WY 450/531
WY 500/531
WY 531/531
WATER-YEAR: PASS


In [12]:
# ============================================================
# 11. SEASONAL / YEAR-SEASON METRICS — Top18 primary
#     EXACT manuscript seasons: JFM, AMJ, JAS, OND.
#     These are fixed calendar-quarter blocks, not DJF/MAM/JJA/SON.
# ============================================================

def season_name(m):
    if m in [1,2,3]: return 'JFM'
    if m in [4,5,6]: return 'AMJ'
    if m in [7,8,9]: return 'JAS'
    return 'OND'

def season_year(ts):
    return ts.year

idx = pd.DataFrame({'date': DATES})
idx['season'] = idx['date'].dt.month.map(season_name)
idx['year'] = idx['date'].dt.year

groups = (
    idx.groupby(['year','season'])['date']
    .agg(['min','max','count'])
    .reset_index()
)
# Keep only complete quarterly seasons. 1989-OND and 1999-JAS are valid;
# 1999-OND is absent because the test period ends on 1999-09-30.
groups = groups[groups['count'] >= 89].copy()

SEASON_P = OUT_PICKLES / 'metrics_by_year_season_top18.p'
rows = []
sp = {}

for bi,b in enumerate(BASINS):
    sp[b] = {}
    o_full = canonical_obs_da(b)
    for s in SEEDS:
        sp[b][s] = {}
        p_full = ens18[b][s]['ensemble']
        for _,g in groups.iterrows():
            yr = int(g['year'])
            seas = g['season']
            dates_sel = idx.loc[(idx['year']==yr) & (idx['season']==seas), 'date']
            o = o_full.sel(date=dates_sel.to_numpy())
            p = p_full.sel(date=dates_sel.to_numpy())
            md = calc_metrics(o, p, DO_CLIP, MAPE_MODE)
            sp[b][s].setdefault(str(yr), {})[seas] = {'Top18': md}
            for k,v in md.items():
                rows.append({
                    'basin':b, 'seed':s, 'year':yr, 'season':seas,
                    'method':'Top18', 'metric':k, 'value':v
                })
    if (bi+1)%50==0 or bi==len(BASINS)-1:
        print(f'Season {bi+1}/{len(BASINS)}')

pd.to_pickle(sp, SEASON_P)
season_top18_long = pd.DataFrame(rows)
save_df(season_top18_long, 'season_top18_long.csv')

# Flatten the existing artifact so old frameworks retain their original values.
old_season = read_pickle_compat(OLD_METRICS_SEASON_P)
flat = flatten_metric_tree(old_season, BASINS, SEEDS)
save_df(flat, 'season_old_flat_audit.csv')
oldrows = []
for _,r in flat.iterrows():
    if r.get('method') is None or pd.isna(r.get('year')) or r.get('season') is None:
        continue
    if r.get('season') not in ['JFM','AMJ','JAS','OND']:
        continue
    for metric in NH_METRIC_LIST + ['MAPE']:
        if metric in r and pd.notna(r[metric]):
            oldrows.append({
                'basin':r['basin'], 'seed':r['seed'], 'year':int(r['year']),
                'season':r['season'], 'method':r['method'],
                'metric':metric, 'value':r[metric]
            })

season_combined = pd.concat(
    [pd.DataFrame(oldrows), season_top18_long],
    ignore_index=True
)
save_df(season_combined, 'season_comparison_long.csv')

# Hierarchical manuscript-style summary:
# 1) median across seeds for each basin×year×season
# 2) median across basins for each year×season
# 3) median across available years for each season
season_basin_year = (
    season_combined.groupby(['basin','year','season','method','metric'], as_index=False)['value']
    .median()
    .rename(columns={'value':'seed_median'})
)
season_year_summary = (
    season_basin_year.groupby(['year','season','method','metric'], as_index=False)['seed_median']
    .median()
    .rename(columns={'seed_median':'basin_median'})
)
season_manuscript_summary = (
    season_year_summary.groupby(['season','method','metric'], as_index=False)['basin_median']
    .median()
    .rename(columns={'basin_median':'season_median_across_years'})
)
save_df(season_basin_year, 'season_basin_year_seed_medians.csv')
save_df(season_year_summary, 'season_year_basin_medians.csv')
save_df(season_manuscript_summary, 'season_manuscript_summary.csv')

# Audit against the published/current-manuscript Top10 & hydrology-informed NSE values.
# Tolerance allows tiny differences from serialization/rounding but catches wrong grouping.
expected_season_nse = {
    'Benchmark':    {'JFM':0.685, 'AMJ':0.696, 'JAS':0.368, 'OND':0.565},
    'Top10':        {'JFM':0.737, 'AMJ':0.751, 'JAS':0.488, 'OND':0.634},
    'Cluster-wise': {'JFM':0.745, 'AMJ':0.761, 'JAS':0.494, 'OND':0.644},
}
season_audit_rows = []
for method, vals in expected_season_nse.items():
    for seas, expected in vals.items():
        d = season_manuscript_summary[
            (season_manuscript_summary['method']==method) &
            (season_manuscript_summary['metric']=='NSE') &
            (season_manuscript_summary['season']==seas)
        ]
        actual = float(d['season_median_across_years'].iloc[0]) if len(d) else np.nan
        season_audit_rows.append({
            'method':method, 'season':seas, 'expected_manuscript':expected,
            'reproduced':actual, 'abs_diff':abs(actual-expected) if np.isfinite(actual) else np.nan
        })
season_audit = pd.DataFrame(season_audit_rows)
save_df(season_audit, 'season_manuscript_reproduction_audit.csv')
display(season_audit)

# Do not stop on small historical rounding/aggregation differences, but clearly flag if material.
if season_audit['abs_diff'].notna().any() and season_audit['abs_diff'].max() > 0.03:
    print('ATTENTION — seasonal summary differs materially from the manuscript reference; '
          'use season_manuscript_reproduction_audit.csv before manuscript editing.')
else:
    print('SEASONAL MANUSCRIPT CONSISTENCY: PASS/NEAR-PASS')

# Table-ready Top18 seasonal NSE summary
season_nse_table = season_manuscript_summary[
    season_manuscript_summary['metric']=='NSE'
].pivot(index='method', columns='season', values='season_median_across_years')
season_nse_table = season_nse_table.reindex(columns=['JFM','AMJ','JAS','OND'])
season_nse_table['Mean of seasons'] = season_nse_table.mean(axis=1)
season_nse_table.to_csv(OUT_TABLES / 'season_NSE_table_manuscript_style.csv')
display(season_nse_table)

del old_season
gc.collect()
print('SEASONAL: PASS')


Season 50/531
Season 100/531


C:\Users\hosseinifh\neuralhydrology\neuralhydrology\evaluation\metrics.py:310: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  r, _ = stats.pearsonr(obs.values, sim.values)
C:\Users\hosseinifh\neuralhydrology\neuralhydrology\evaluation\metrics.py:346: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  r, _ = stats.pearsonr(obs.values, sim.values)
C:\Users\hosseinifh\neuralhydrology\neuralhydrology\evaluation\metrics.py:467: RuntimeWarning: divide by zero encountered in scalar divide
  fhv = np.sum(sim - obs) / np.sum(obs)
C:\Users\hosseinifh\neuralhydrology\neuralhydrology\evaluation\metrics.py:310: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  r, _ = stats.pearsonr(obs.values, sim.values)
C:\Users\hosseinifh\neuralhydrology\neuralhydrology\evaluation\metrics.py:346: ConstantInputWarning: An input array is constant; the correlation coefficient i

Season 150/531
Season 200/531
Season 250/531


C:\Users\hosseinifh\neuralhydrology\neuralhydrology\evaluation\metrics.py:310: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  r, _ = stats.pearsonr(obs.values, sim.values)
C:\Users\hosseinifh\neuralhydrology\neuralhydrology\evaluation\metrics.py:346: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  r, _ = stats.pearsonr(obs.values, sim.values)
C:\Users\hosseinifh\neuralhydrology\neuralhydrology\evaluation\metrics.py:467: RuntimeWarning: divide by zero encountered in scalar divide
  fhv = np.sum(sim - obs) / np.sum(obs)
C:\Users\hosseinifh\neuralhydrology\neuralhydrology\evaluation\metrics.py:310: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  r, _ = stats.pearsonr(obs.values, sim.values)
C:\Users\hosseinifh\neuralhydrology\neuralhydrology\evaluation\metrics.py:346: ConstantInputWarning: An input array is constant; the correlation coefficient i

Season 300/531


C:\Users\hosseinifh\neuralhydrology\neuralhydrology\evaluation\metrics.py:310: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  r, _ = stats.pearsonr(obs.values, sim.values)
C:\Users\hosseinifh\neuralhydrology\neuralhydrology\evaluation\metrics.py:346: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  r, _ = stats.pearsonr(obs.values, sim.values)
C:\Users\hosseinifh\neuralhydrology\neuralhydrology\evaluation\metrics.py:467: RuntimeWarning: divide by zero encountered in scalar divide
  fhv = np.sum(sim - obs) / np.sum(obs)
C:\Users\hosseinifh\neuralhydrology\neuralhydrology\evaluation\metrics.py:310: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  r, _ = stats.pearsonr(obs.values, sim.values)
C:\Users\hosseinifh\neuralhydrology\neuralhydrology\evaluation\metrics.py:346: ConstantInputWarning: An input array is constant; the correlation coefficient i

... (7 outputs left out)

,method,season,expected_manuscript,reproduced,abs_diff
0,Benchmark,JFM,0.685,0.700125,0.015125
1,Benchmark,AMJ,0.696,0.722519,0.026519
2,Benchmark,JAS,0.368,0.423073,0.055073
3,Benchmark,OND,0.565,0.579588,0.014588
4,Top10,JFM,0.737,0.740323,0.003323
5,Top10,AMJ,0.751,0.753280,0.002280
6,Top10,JAS,0.488,0.495824,0.007824
7,Top10,OND,0.634,0.638639,0.004639
8,Cluster-wise,JFM,0.745,0.747493,0.002493
9,Cluster-wise,AMJ,0.761,0.764133,0.003133


ATTENTION — seasonal summary differs materially from the manuscript reference; use season_manuscript_reproduction_audit.csv before manuscript editing.


season,JFM,AMJ,JAS,OND,Mean of seasons
method,,,,,
Benchmark,0.700125,0.722519,0.423073,0.579588,0.606326
Cluster-wise,0.747493,0.764133,0.502435,0.650873,0.666234
Top10,0.740323,0.753280,0.495824,0.638639,0.657017
Top18,0.751827,0.759675,0.503191,0.638179,0.663218


SEASONAL: PASS


In [13]:
def discover_cluster_map(root,basins):
    bset=set(basins); audits=[]
    for p in root.rglob('*'):
        if not p.is_file() or 'cluster' not in p.name.lower() or p.suffix.lower() not in {'.csv','.txt','.p','.pkl','.pickle'} or p.stat().st_size>150*1024**2: continue
        try:
            if p.suffix.lower()=='.csv': df=pd.read_csv(p)
            elif p.suffix.lower()=='.txt': df=pd.read_csv(p,sep=None,engine='python')
            else:
                obj=pd.read_pickle(p)
                if isinstance(obj,pd.DataFrame): df=obj.copy()
                elif isinstance(obj,dict) and set(map(str,obj)).intersection(bset): df=pd.DataFrame({'basin':list(map(str,obj.keys())),'cluster':list(obj.values())})
                else: continue
            best=None; ov=0
            for c in df.columns:
                vals=set(df[c].astype(str).str.zfill(8)); n=len(vals&bset)
                if n>ov: best,ov=c,n
            cc=[c for c in df.columns if 'cluster' in str(c).lower()]; audits.append((str(p),best,ov,cc))
            if ov>=500 and cc:
                out=df[[best,cc[0]]].copy(); out.columns=['basin','cluster']; out['basin']=out['basin'].astype(str).str.zfill(8); out=out[out['basin'].isin(bset)].drop_duplicates('basin')
                if len(out)>=500: return out,audits
        except Exception: pass
    return None,audits
cluster_map,cluster_audit=discover_cluster_map(PROJECT,BASINS)
pd.DataFrame(cluster_audit,columns=['path','basin_col','overlap','cluster_cols']).to_csv(OUT_AUDIT/'cluster_mapping_candidates.csv',index=False)
if cluster_map is not None:
    save_df(cluster_map,'basin_cluster_map_used.csv'); cluster_metrics=basin_method.merge(cluster_map,on='basin',how='inner'); save_df(cluster_metrics,'cluster_stratified_basin_metrics_long.csv')
    cluster_summary=cluster_metrics.groupby(['cluster','method','metric'])['basin_seed_median'].agg(['count','median','mean','std']).reset_index(); save_df(cluster_summary,'cluster_stratified_summary.csv'); print('CLUSTER MAP: PASS',len(cluster_map)); display(cluster_map['cluster'].value_counts().sort_index())
else:
    cluster_metrics=None; cluster_summary=None; print('ATTENTION — basin→cluster map not auto-detected; global Cluster-wise method comparison is still complete.')


CLUSTER MAP: PASS 531


cluster
1     83
2    195
3     67
4     61
5     66
6     59
Name: count, dtype: int64

In [14]:
def detect_events():
    for p in [HIGHFLOW_LOG,GLOBAL_EVENT_CATALOG]:
        if not p.exists(): continue
        try: df=pd.read_csv(p)
        except Exception: continue
        cols=list(df.columns); low={c:str(c).lower() for c in cols}
        bcol=next((c for c in cols if any(x in low[c] for x in ['basin','gauge','gage'])),None)
        if bcol is None:
            best=None; ov=0
            for c in cols:
                n=len(set(df[c].astype(str).str.zfill(8))&set(BASINS))
                if n>ov: best,ov=c,n
            if ov>100: bcol=best
        scol=next((c for c in cols if 'start' in low[c] and any(x in low[c] for x in ['date','time'])),None)
        ecol=next((c for c in cols if 'end' in low[c] and any(x in low[c] for x in ['date','time'])),None)
        pcol=next((c for c in cols if 'peak' in low[c] and any(x in low[c] for x in ['date','time'])),None)
        if bcol is not None and ((scol and ecol) or pcol):
            out=df.copy(); out['basin']=out[bcol].astype(str).str.zfill(8); out=out[out['basin'].isin(BASINS)].copy()
            if scol and ecol: out['event_start']=pd.to_datetime(out[scol],errors='coerce'); out['event_end']=pd.to_datetime(out[ecol],errors='coerce')
            else:
                peak=pd.to_datetime(out[pcol],errors='coerce'); out['event_start']=peak-pd.Timedelta(days=3); out['event_end']=peak+pd.Timedelta(days=3)
            out=out.dropna(subset=['event_start','event_end']); out=out[(out['event_end']>=DATES.min())&(out['event_start']<=DATES.max())].copy(); out['event_id']=np.arange(len(out)).astype(str)
            return out,p,{'basin_col':bcol,'start_col':scol,'end_col':ecol,'peak_col':pcol}
    return None,None,None
events,event_source,event_schema=detect_events()
if events is None:
    highflow_top18=pd.DataFrame(); print('ATTENTION — event windows not detected; high-flow Top18 skipped.')
else:
    print('Event source:',event_source,'schema:',event_schema,'N=',len(events)); events.to_csv(OUT_AUDIT/'highflow_events_used.csv',index=False); rows=[]
    for n,(_,r) in enumerate(events.iterrows()):
        b=r['basin']; start=max(pd.Timestamp(r['event_start']),DATES.min()); end=min(pd.Timestamp(r['event_end']),DATES.max()); o=canonical_obs_da(b).sel(date=slice(start,end))
        if o.sizes.get('date',0)<3: continue
        for s in SEEDS:
            p=ens18[b][s]['ensemble'].sel(date=slice(start,end)); md=calc_metrics(o,p,DO_CLIP,MAPE_MODE)
            for k,v in md.items(): rows.append({'event_id':r['event_id'],'basin':b,'seed':s,'event_start':start,'event_end':end,'method':'Top18','metric':k,'value':v})
        if (n+1)%250==0: print(f'Events {n+1}/{len(events)}')
    highflow_top18=pd.DataFrame(rows); save_df(highflow_top18,'highflow_eventwise_top18_long.csv'); print('HIGH-FLOW EVENT ANALYSIS: PASS')


ATTENTION — event windows not detected; high-flow Top18 skipped.


In [ ]:
normalized_bins_status={'old_artifact_exists':OLD_HIGHFLOW_BINS_P.exists(),'exact_top18_reproduction':False,'reason':None}
if OLD_HIGHFLOW_BINS_P.exists():
    size_gb=OLD_HIGHFLOW_BINS_P.stat().st_size/1024**3; normalized_bins_status['old_size_gb']=size_gb
    normalized_bins_status['reason']='Exact Top18 normalized-bin extension not auto-generated because the original bin-construction code is not present in the supplied training notebook. Existing artifact preserved; no new bin definition is invented.'
    print('ATTENTION —',normalized_bins_status['reason']); print('Existing:',OLD_HIGHFLOW_BINS_P,f'({size_gb:.2f} GB)')
else: normalized_bins_status['reason']='Old normalized-bin artifact not found.'
(OUT_AUDIT/'normalized_bins_status.json').write_text(json.dumps(normalized_bins_status,indent=2),encoding='utf-8')


ATTENTION — Exact Top18 normalized-bin extension not auto-generated because the original bin-construction code is not present in the supplied training notebook. Existing artifact preserved; no new bin definition is invented.
Existing: F:\Experiments\CAMELS_US\Clean_4_upload\Pickles\metrics_highflow_events_normalized_bins.p (1.15 GB)


335

: 

In [ ]:
plot_methods=[m for m in ['Top18','Top10','Cluster-wise','Benchmark','Top18-mean'] if m in basin_method['method'].unique()]
for metric in KEY_METRICS:
    d=basin_method[(basin_method['metric']==metric)&(basin_method['method'].isin(plot_methods))]
    if d.empty: continue
    arrays=[d.loc[d['method']==m,'basin_seed_median'].dropna().to_numpy() for m in plot_methods]
    fig,ax=plt.subplots(figsize=(8.5,5.3)); ax.boxplot(arrays,labels=plot_methods,showfliers=False); ax.set_ylabel(metric); ax.set_title(f'Overall test-period performance — {metric}'); ax.grid(axis='y',alpha=.25); fig.tight_layout(); figure_save(fig,f"overall_boxplot_{metric.replace('/','_')}")
    a=basin_method[(basin_method['method']=='Top18')&(basin_method['metric']==metric)][['basin','basin_seed_median']].rename(columns={'basin_seed_median':'Top18'}); b=basin_method[(basin_method['method']=='Top10')&(basin_method['metric']==metric)][['basin','basin_seed_median']].rename(columns={'basin_seed_median':'Top10'}); mm=a.merge(b,on='basin')
    if not mm.empty:
        fig,ax=plt.subplots(figsize=(7.5,4.8)); ax.hist((mm['Top18']-mm['Top10']).dropna().to_numpy(),bins=35); ax.axvline(0,linewidth=1); ax.set_xlabel(f'Top18 − Top10 ({metric})'); ax.set_ylabel('Number of basins'); ax.set_title(f'Paired basin-wise change — {metric}'); ax.grid(axis='y',alpha=.25); fig.tight_layout(); figure_save(fig,f"delta_top18_minus_top10_{metric.replace('/','_')}")
print('Overall figures complete.')


C:\Users\hosseinifh\AppData\Local\Temp\ipykernel_13340\822853548.py:6: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  fig,ax=plt.subplots(figsize=(8.5,5.3)); ax.boxplot(arrays,labels=plot_methods,showfliers=False); ax.set_ylabel(metric); ax.set_title(f'Overall test-period performance — {metric}'); ax.grid(axis='y',alpha=.25); fig.tight_layout(); figure_save(fig,f"overall_boxplot_{metric.replace('/','_')}")


In [ ]:
if not wy_combined.empty:
    wy_summary=wy_combined[wy_combined['metric'].isin(['NSE','KGE'])].groupby(['year','method','metric'],as_index=False)['value'].median(); save_df(wy_summary,'water_year_summary_median.csv')
    for metric in ['NSE','KGE']:
        d=wy_summary[wy_summary['metric']==metric]; methods=[m for m in ['Top18','Top10','Cluster-wise','Benchmark'] if m in d['method'].unique()]; fig,ax=plt.subplots(figsize=(8.5,5))
        for m in methods:
            dm=d[d['method']==m].sort_values('year'); ax.plot(dm['year'],dm['value'],marker='o',label=m)
        ax.set_xlabel('Water year'); ax.set_ylabel(f'Median {metric}'); ax.set_title(f'Water-year performance — {metric}'); ax.legend(); ax.grid(alpha=.25); fig.tight_layout(); figure_save(fig,f'water_year_{metric}')
if not season_combined.empty:
    # Use the hierarchical summary created in the seasonal cell, matching manuscript seasons.
    season_summary = season_manuscript_summary[
        season_manuscript_summary['metric'].isin(['NSE','KGE','RMSE','Peak-Timing'])
    ].copy()
    save_df(season_summary, 'season_summary_median.csv')
    order = ['JFM','AMJ','JAS','OND']
    for metric in ['NSE','KGE','RMSE','Peak-Timing']:
        d = season_summary[season_summary['metric']==metric]
        if d.empty:
            continue
        methods = [m for m in ['Top18','Top10','Cluster-wise','Benchmark'] if m in d['method'].unique()]
        x = np.arange(len(order))
        width = .8/max(len(methods),1)
        fig,ax = plt.subplots(figsize=(8.5,5))
        for i,m in enumerate(methods):
            dm = d[d['method']==m].set_index('season').reindex(order)
            ax.bar(
                x+(i-(len(methods)-1)/2)*width,
                dm['season_median_across_years'],
                width=width, label=m
            )
        ax.set_xticks(x)
        ax.set_xticklabels(order)
        ax.set_ylabel(f'Median {metric}')
        ax.set_title(f'Seasonal performance — {metric}')
        ax.legend()
        ax.grid(axis='y',alpha=.25)
        fig.tight_layout()
        figure_save(fig, f"season_{metric.replace('/','_')}")

    # Top18 seasonal heatmap across years (same conceptual view as manuscript Figure 12).
    d = season_year_summary[
        (season_year_summary['method']=='Top18') &
        (season_year_summary['metric']=='NSE')
    ].copy()
    if not d.empty:
        heat = d.pivot(index='year', columns='season', values='basin_median').reindex(columns=order)
        fig,ax = plt.subplots(figsize=(7.4,6.0))
        im = ax.imshow(heat.to_numpy(), aspect='auto')
        ax.set_xticks(np.arange(len(order))); ax.set_xticklabels(order)
        ax.set_yticks(np.arange(len(heat.index))); ax.set_yticklabels(heat.index.astype(int))
        ax.set_xlabel('Season'); ax.set_ylabel('Year')
        ax.set_title('Top18 seasonal median basin-wise NSE across years')
        fig.colorbar(im, ax=ax, label='NSE')
        fig.tight_layout()
        figure_save(fig, 'season_Top18_NSE_heatmap_by_year')

if cluster_summary is not None:
    for metric in ['NSE','KGE','RMSE','Peak-Timing']:
        d=cluster_summary[cluster_summary['metric']==metric]
        if d.empty: continue
        clusters=list(pd.unique(d['cluster'])); methods=[m for m in ['Top18','Top10','Cluster-wise','Benchmark'] if m in d['method'].unique()]; x=np.arange(len(clusters)); width=.8/max(len(methods),1); fig,ax=plt.subplots(figsize=(max(8.5,len(clusters)*1.15),5))
        for i,m in enumerate(methods):
            dm=d[d['method']==m].set_index('cluster').reindex(clusters); ax.bar(x+(i-(len(methods)-1)/2)*width,dm['median'],width=width,label=m)
        ax.set_xticks(x); ax.set_xticklabels([str(c) for c in clusters]); ax.set_xlabel('Hydrological cluster'); ax.set_ylabel(f'Median {metric}'); ax.set_title(f'Cluster-stratified performance — {metric}'); ax.legend(); ax.grid(axis='y',alpha=.25); fig.tight_layout(); figure_save(fig,f"cluster_{metric.replace('/','_')}")
print('Temporal/cluster figures complete.')


In [ ]:
if 'highflow_top18' in globals() and not highflow_top18.empty:
    hf_summary=highflow_top18.groupby('metric')['value'].agg(['count','median','mean','std']).reset_index(); save_df(hf_summary,'highflow_top18_summary.csv')
    for metric in ['NSE','KGE','FHV','Peak-Timing','Peak-MAPE','Missed-Peaks']:
        d=highflow_top18[highflow_top18['metric']==metric]['value'].dropna()
        if d.empty: continue
        fig,ax=plt.subplots(figsize=(7.2,4.6)); ax.hist(d.to_numpy(),bins=35); ax.set_xlabel(metric); ax.set_ylabel('Event × seed count'); ax.set_title(f'Top18 high-flow event-wise distribution — {metric}'); ax.grid(axis='y',alpha=.25); fig.tight_layout(); figure_save(fig,f"highflow_top18_{metric.replace('/','_')}")
print('High-flow figures complete.')


In [ ]:
manifest=[]
for group,folder in [('pickles',OUT_PICKLES),('tables',OUT_TABLES),('figures',OUT_FIGURES),('audit',OUT_AUDIT)]:
    for p in sorted(folder.rglob('*')):
        if p.is_file(): manifest.append({'group':group,'relative_path':str(p.relative_to(OUT)),'size_MB':round(p.stat().st_size/1024**2,3)})
manifest=pd.DataFrame(manifest); manifest.to_csv(OUT/'manifest.csv',index=False)
headline=paired_summary[(paired_summary['baseline'].isin(['Top10','Cluster-wise','Benchmark']))&(paired_summary['metric'].isin(['NSE','KGE','RMSE','Peak-Timing','Peak-MAPE','Missed-Peaks']))].copy(); headline.to_csv(OUT_TABLES/'HEADLINE_Top18_comparison.csv',index=False)
print('='*100); print('TOP18 FINAL ANALYSIS COMPLETE'); print('='*100); print('Output:',OUT); print('Files written:',len(manifest)); display(headline)
if not normalized_bins_status.get('exact_top18_reproduction'):
    print('\nATTENTION — normalized high-flow bin analysis was not silently reconstructed.')
    print(normalized_bins_status.get('reason'))


In [4]:
# =====================================================================
# FINAL REVIEW PACKAGE FOR CHATGPT
# Run this cell after the full Top18 analysis has completed.
# =====================================================================

from pathlib import Path
import pandas as pd
import numpy as np
import json
import zipfile
import shutil
from datetime import datetime

# ---------------------------------------------------------------------
# Paths
# ---------------------------------------------------------------------

REPORT_MD = OUT / "REPORT_FOR_CHATGPT.md"
REPORT_JSON = OUT / "REPORT_DATA_FOR_CHATGPT.json"
REVIEW_DIR = OUT / "Review_Package"
REVIEW_ZIP = OUT / "CAMELS_Top18_Final_Review_Package.zip"

if REVIEW_DIR.exists():
    shutil.rmtree(REVIEW_DIR)

REVIEW_DIR.mkdir(parents=True, exist_ok=True)

# ---------------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------------

def read_csv_safe(name):
    p = OUT_TABLES / name
    if p.exists():
        try:
            return pd.read_csv(p)
        except Exception as e:
            print(f"WARNING reading {name}: {e}")
    return None


def df_to_md(df, max_rows=50, decimals=5):
    if df is None or df.empty:
        return "_Not available._"

    x = df.copy()

    for c in x.select_dtypes(include=[np.number]).columns:
        x[c] = x[c].round(decimals)

    if len(x) > max_rows:
        x = x.head(max_rows)

    try:
        return x.to_markdown(index=False)
    except Exception:
        return x.to_string(index=False)


def metric_direction(metric):
    higher = {"NSE", "KGE", "Pearson-r"}
    lower = {
        "MSE", "RMSE", "Peak-Timing",
        "Peak-MAPE", "Missed-Peaks", "MAPE"
    }
    zero_best = {"FHV", "FMS", "FLV", "Beta-NSE"}

    if metric in higher:
        return "higher"
    if metric in lower:
        return "lower"
    if metric in zero_best:
        return "zero"
    return "context"


def safe_num(x):
    try:
        if pd.isna(x):
            return None
        return float(x)
    except Exception:
        return None


# ---------------------------------------------------------------------
# Load main result tables
# ---------------------------------------------------------------------

paired = read_csv_safe("paired_comparison_summary.csv")
overall = read_csv_safe("overall_summary_by_method_metric.csv")
basin_metrics = read_csv_safe("overall_basin_seed_medians.csv")

cluster_summary_df = read_csv_safe("cluster_stratified_summary.csv")
cluster_map_df = read_csv_safe("basin_cluster_map_used.csv")

wy_summary_df = read_csv_safe("water_year_summary_median.csv")
season_summary_df = read_csv_safe("season_summary_median.csv")

hf_summary_df = read_csv_safe("highflow_top18_summary.csv")

protocol_df = read_csv_safe("metric_protocol_candidates.csv")
protocol_samples = read_csv_safe("metric_protocol_audit_samples.csv")

headline = read_csv_safe("HEADLINE_Top18_comparison.csv")

# ---------------------------------------------------------------------
# Core metric list
# ---------------------------------------------------------------------

CORE_METRICS = [
    "NSE",
    "KGE",
    "RMSE",
    "Pearson-r",
    "FHV",
    "FMS",
    "FLV",
    "Peak-Timing",
    "Peak-MAPE",
    "Missed-Peaks",
    "MAPE",
]

# ---------------------------------------------------------------------
# Extract primary Top18-vs-Top10 findings
# ---------------------------------------------------------------------

primary_rows = []

if paired is not None and not paired.empty:

    p = paired[
        (paired["baseline"] == "Top10") &
        (paired["metric"].isin(CORE_METRICS))
    ].copy()

    for _, r in p.iterrows():

        metric = r["metric"]

        primary_rows.append({
            "metric": metric,
            "direction": metric_direction(metric),
            "Top18_median": safe_num(r.get("median_top18")),
            "Top10_median": safe_num(r.get("median_baseline")),
            "raw_delta_Top18_minus_Top10":
                safe_num(r.get("median_raw_delta_top18_minus_baseline")),
            "pct_basins_improved":
                safe_num(r.get("pct_basins_improved")),
            "wilcoxon_p":
                safe_num(r.get("wilcoxon_p")),
        })

primary_df = pd.DataFrame(primary_rows)


# ---------------------------------------------------------------------
# Seed-level comparison
# ---------------------------------------------------------------------

seed_summary = None

overall_long_p = OUT_TABLES / "overall_metrics_long.csv"

if overall_long_p.exists():

    ol = pd.read_csv(overall_long_p)

    seed_core = ol[
        ol["metric"].isin(["NSE", "KGE", "RMSE"])
        & ol["method"].isin(["Top18", "Top10"])
    ].copy()

    seed_summary = (
        seed_core
        .groupby(["seed", "method", "metric"], as_index=False)["value"]
        .median()
    )

    pivot = seed_summary.pivot_table(
        index=["seed", "metric"],
        columns="method",
        values="value"
    ).reset_index()

    if {"Top18", "Top10"}.issubset(pivot.columns):

        pivot["delta_Top18_minus_Top10"] = (
            pivot["Top18"] - pivot["Top10"]
        )

        pivot.to_csv(
            REVIEW_DIR / "seed_level_Top18_vs_Top10.csv",
            index=False
        )

        seed_summary_report = pivot

    else:
        seed_summary_report = None

else:
    seed_summary_report = None


# ---------------------------------------------------------------------
# Overall ranking by important metrics
# ---------------------------------------------------------------------

ranking_tables = {}

if overall is not None:

    for metric in ["NSE", "KGE", "RMSE", "Peak-Timing", "Peak-MAPE"]:

        d = overall[overall["metric"] == metric].copy()

        if d.empty:
            continue

        ascending = metric_direction(metric) == "lower"

        d = d.sort_values(
            "median",
            ascending=ascending
        )

        ranking_tables[metric] = d


# ---------------------------------------------------------------------
# Best / worst basin-level Top18 changes
# ---------------------------------------------------------------------

basin_delta_tables = {}

if basin_metrics is not None:

    for metric in ["NSE", "KGE", "RMSE"]:

        d = basin_metrics[
            basin_metrics["metric"] == metric
        ]

        a = d[
            d["method"] == "Top18"
        ][["basin", "basin_seed_median"]].rename(
            columns={"basin_seed_median": "Top18"}
        )

        b = d[
            d["method"] == "Top10"
        ][["basin", "basin_seed_median"]].rename(
            columns={"basin_seed_median": "Top10"}
        )

        m = a.merge(b, on="basin")

        if m.empty:
            continue

        m["delta"] = m["Top18"] - m["Top10"]

        if metric_direction(metric) == "lower":
            m["improvement"] = -m["delta"]
        else:
            m["improvement"] = m["delta"]

        basin_delta_tables[metric] = {
            "best": m.sort_values(
                "improvement",
                ascending=False
            ).head(10),
            "worst": m.sort_values(
                "improvement",
                ascending=True
            ).head(10),
        }


# ---------------------------------------------------------------------
# Detect audit warnings
# ---------------------------------------------------------------------

audit_messages = []

for p in sorted(OUT_AUDIT.glob("*")):

    if p.suffix.lower() in {".json", ".txt", ".log"}:

        try:
            txt = p.read_text(
                encoding="utf-8",
                errors="replace"
            )

            if "ATTENTION" in txt.upper():
                audit_messages.append(
                    f"{p.name}: contains ATTENTION marker."
                )

        except Exception:
            pass


normalized_status_p = (
    OUT_AUDIT / "normalized_bins_status.json"
)

normalized_status = None

if normalized_status_p.exists():

    try:
        normalized_status = json.loads(
            normalized_status_p.read_text(
                encoding="utf-8"
            )
        )
    except Exception:
        pass


# ---------------------------------------------------------------------
# JSON machine-readable summary
# ---------------------------------------------------------------------

json_report = {

    "generated": datetime.now().isoformat(),

    "experiment": {
        "name": "CAMELS-US Top18 control",
        "primary_comparison": "Top18-median vs Top10-median",
        "secondary_comparison": "Top18-mean",
        "n_basins": 531,
        "n_seeds": 10,
        "top10_members": 10,
        "top18_members": 18,
        "test_start": "1989-10-01",
        "test_end": "1999-09-30",
        "test_days": 3652,
    },

    "primary_results":
        primary_df.to_dict(orient="records")
        if not primary_df.empty
        else [],

    "normalized_highflow_bins":
        normalized_status,

    "audit_messages":
        audit_messages,
}

REPORT_JSON.write_text(
    json.dumps(
        json_report,
        indent=2,
        ensure_ascii=False
    ),
    encoding="utf-8"
)


# ---------------------------------------------------------------------
# Build Markdown report
# ---------------------------------------------------------------------

parts = []

parts.append(
f"""# CAMELS-US 2026 — Top18 Final Control Experiment

Generated: **{datetime.now().strftime('%Y-%m-%d %H:%M')}**

## 1. Experiment design

- Basins: **531**
- Random seeds: **10**
- Existing compact ensemble: **Top10 = 10 members**
- Extended control ensemble: **Top18 = 18 members**
- Primary aggregation: **median**
- Secondary Top18 aggregation: **mean**
- Test period: **1989-10-01 → 1999-09-30**
- Test length: **3652 daily observations**
- Canonical observations: `Obs_test.p`
- Primary scientific comparison: **Top18-median vs Top10-median**
- Additional comparisons: **Hydrology-informed / Cluster-wise** and **Benchmark**

---

# 2. Headline Top18 vs Top10 results

"""
)

parts.append(df_to_md(primary_df, max_rows=100))

parts.append(
"""

Interpretation rule:

- NSE, KGE, Pearson-r: higher is better.
- RMSE, MSE, MAPE, Peak-MAPE, Peak-Timing and Missed-Peaks: lower is better.
- FHV/FMS/FLV are bias-type measures; values closer to zero are preferable.

---

# 3. Full headline comparison table

"""
)

parts.append(
    df_to_md(
        headline,
        max_rows=100
    )
)

parts.append(
"""

---

# 4. Overall method ranking

"""
)

for metric, tab in ranking_tables.items():

    parts.append(
        f"\n## {metric}\n"
    )

    cols = [
        c for c in [
            "method",
            "count",
            "median",
            "mean",
            "std",
            "min",
            "max",
        ]
        if c in tab.columns
    ]

    parts.append(
        df_to_md(
            tab[cols],
            max_rows=20
        )
    )


# ---------------------------------------------------------------------
# Seed stability
# ---------------------------------------------------------------------

parts.append(
"""

---

# 5. Stability across the 10 random seeds

"""
)

if seed_summary_report is not None:

    parts.append(
        df_to_md(
            seed_summary_report,
            max_rows=100
        )
    )

else:

    parts.append(
        "_Seed-level summary unavailable._"
    )


# ---------------------------------------------------------------------
# Basin gains / losses
# ---------------------------------------------------------------------

parts.append(
"""

---

# 6. Basin-wise sensitivity to increasing ensemble size

"""
)

for metric, tabs in basin_delta_tables.items():

    parts.append(
        f"\n## {metric} — strongest improvements\n"
    )

    parts.append(
        df_to_md(
            tabs["best"],
            max_rows=10
        )
    )

    parts.append(
        f"\n## {metric} — strongest deteriorations\n"
    )

    parts.append(
        df_to_md(
            tabs["worst"],
            max_rows=10
        )
    )


# ---------------------------------------------------------------------
# Hydrological cluster comparison
# ---------------------------------------------------------------------

parts.append(
"""

---

# 7. Hydrological-cluster / hydrology-informed analysis

"""
)

if cluster_summary_df is not None:

    cluster_core = cluster_summary_df[
        cluster_summary_df["metric"].isin(
            ["NSE", "KGE", "RMSE", "Peak-Timing"]
        )
    ]

    parts.append(
        df_to_md(
            cluster_core,
            max_rows=200
        )
    )

else:

    parts.append(
        "_No basin→cluster mapping was available for stratified analysis. "
        "Global comparison against the Cluster-wise hydrology-informed method "
        "is still included in the overall tables._"
    )


# ---------------------------------------------------------------------
# Water-year
# ---------------------------------------------------------------------

parts.append(
"""

---

# 8. Water-year analysis

"""
)

parts.append(
    df_to_md(
        wy_summary_df,
        max_rows=200
    )
)


# ---------------------------------------------------------------------
# Seasons
# ---------------------------------------------------------------------

parts.append(
"""

---

# 9. Seasonal analysis

"""
)

parts.append(
    df_to_md(
        season_summary_df,
        max_rows=200
    )
)


# ---------------------------------------------------------------------
# High flow
# ---------------------------------------------------------------------

parts.append(
"""

---

# 10. High-flow event analysis

"""
)

parts.append(
    df_to_md(
        hf_summary_df,
        max_rows=100
    )
)


# ---------------------------------------------------------------------
# Metric protocol
# ---------------------------------------------------------------------

parts.append(
"""

---

# 11. Metric-protocol reproduction audit

Before computing the new Top18 metrics, the pipeline attempts to reproduce
the existing Top10 metrics using the same canonical observations.

"""
)

parts.append(
    df_to_md(
        protocol_df,
        max_rows=50
    )
)


# ---------------------------------------------------------------------
# Audit limitations
# ---------------------------------------------------------------------

parts.append(
"""

---

# 12. Audit / unresolved items

"""
)

if normalized_status is not None:

    parts.append(
        "## Normalized high-flow bins\n\n"
        "```json\n"
        + json.dumps(
            normalized_status,
            indent=2,
            ensure_ascii=False
        )
        + "\n```\n"
    )

if audit_messages:

    for x in audit_messages:
        parts.append(f"- {x}")

else:

    parts.append(
        "No additional ATTENTION markers detected in the audit files."
    )


# ---------------------------------------------------------------------
# File inventory
# ---------------------------------------------------------------------

parts.append(
"""

---

# 13. Files available for detailed review

The accompanying ZIP intentionally excludes giant raw pickles and cache
arrays. It contains the numerical tables, audit files and figures needed
to assess the scientific result.

"""
)

REPORT_MD.write_text(
    "\n".join(parts),
    encoding="utf-8"
)


# ---------------------------------------------------------------------
# Copy report + tables + audits + figures to review directory
# ---------------------------------------------------------------------

shutil.copy2(
    REPORT_MD,
    REVIEW_DIR / REPORT_MD.name
)

shutil.copy2(
    REPORT_JSON,
    REVIEW_DIR / REPORT_JSON.name
)


for src_dir, subfolder in [
    (OUT_TABLES, "tables"),
    (OUT_AUDIT, "audit"),
    (OUT_FIGURES, "figures"),
]:

    dest = REVIEW_DIR / subfolder
    dest.mkdir(parents=True, exist_ok=True)

    for p in src_dir.iterdir():

        if not p.is_file():
            continue

        # Avoid accidentally packaging huge binary artifacts.
        if p.stat().st_size > 100 * 1024**2:
            continue

        shutil.copy2(
            p,
            dest / p.name
        )


# ---------------------------------------------------------------------
# Manifest
# ---------------------------------------------------------------------

manifest_rows = []

for p in REVIEW_DIR.rglob("*"):

    if p.is_file():

        manifest_rows.append({
            "file":
                str(
                    p.relative_to(
                        REVIEW_DIR
                    )
                ),
            "size_MB":
                round(
                    p.stat().st_size
                    / 1024**2,
                    3
                )
        })

manifest = pd.DataFrame(
    manifest_rows
).sort_values("file")

manifest.to_csv(
    REVIEW_DIR / "PACKAGE_MANIFEST.csv",
    index=False
)


# ---------------------------------------------------------------------
# ZIP
# ---------------------------------------------------------------------

if REVIEW_ZIP.exists():
    REVIEW_ZIP.unlink()

with zipfile.ZipFile(
    REVIEW_ZIP,
    "w",
    compression=zipfile.ZIP_DEFLATED
) as z:

    for p in REVIEW_DIR.rglob("*"):

        if p.is_file():

            z.write(
                p,
                arcname=str(
                    p.relative_to(
                        REVIEW_DIR
                    )
                )
            )


# ---------------------------------------------------------------------
# Final output
# ---------------------------------------------------------------------

print("=" * 100)
print("FINAL REVIEW PACKAGE CREATED")
print("=" * 100)

print("\nMarkdown report:")
print(REPORT_MD)

print("\nMachine-readable summary:")
print(REPORT_JSON)

print("\nZIP package:")
print(REVIEW_ZIP)

print(
    "\nZIP size:",
    round(
        REVIEW_ZIP.stat().st_size
        / 1024**2,
        2
    ),
    "MB"
)

print("\nPackage files:", len(manifest))

display(manifest)

print("\n" + "=" * 100)
print(
    "UPLOAD TO CHATGPT:"
    "\n1. REPORT_FOR_CHATGPT.md"
    "\n2. CAMELS_Top18_Final_Review_Package.zip"
)
print("=" * 100)

FINAL REVIEW PACKAGE CREATED

Markdown report:
F:\Experiments\CAMELS_US\Clean_4_upload\Revision_2026_Top18_4Framework\Top18_vs_Top10_Final_Analysis\REPORT_FOR_CHATGPT.md

Machine-readable summary:
F:\Experiments\CAMELS_US\Clean_4_upload\Revision_2026_Top18_4Framework\Top18_vs_Top10_Final_Analysis\REPORT_DATA_FOR_CHATGPT.json

ZIP package:
F:\Experiments\CAMELS_US\Clean_4_upload\Revision_2026_Top18_4Framework\Top18_vs_Top10_Final_Analysis\CAMELS_Top18_Final_Review_Package.zip

ZIP size: 49.05 MB

Package files: 22


,file,size_MB
0,REPORT_DATA_FOR_CHATGPT.json,0.004
1,REPORT_FOR_CHATGPT.md,0.018
3,audit\cluster_mapping_candidates.csv,0.000
4,audit\convert_legacy_top10.log,0.007
5,audit\normalized_bins_status.json,0.000
2,seed_level_Top18_vs_Top10.csv,0.002
6,tables\basin_cluster_map_used.csv,0.006
7,tables\cluster_stratified_basin_metrics_long.csv,1.785
8,tables\cluster_stratified_summary.csv,0.034
9,tables\metric_protocol_audit_samples.csv,0.004



UPLOAD TO CHATGPT:
1. REPORT_FOR_CHATGPT.md
2. CAMELS_Top18_Final_Review_Package.zip
